In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

# Gunakan GPU CUDA jika tersedia, atau CPU jika tidak.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(device)}")

In [ ]:
# 1. Baca train dan test dari file NPZ.
with np.load("data/mnist.npz", allow_pickle=False) as data:
    print("Array dalam NPZ:", data.files)

    x_train_original = data["x_train"]
    y_train_original = data["y_train"]
    x_test = data["x_test"]
    y_test = data["y_test"]

# Periksa ukuran dataset MNIST standar.
if x_train_original.shape != (60000, 28, 28):
    raise ValueError("Training harus berukuran (60000, 28, 28).")

if x_test.shape != (10000, 28, 28):
    raise ValueError("Test harus berukuran (10000, 28, 28).")

In [ ]:
# 2. Normalisasi piksel dari 0–255 menjadi 0–1.
x_train_original = x_train_original.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

# Tambahkan satu kanal karena gambar grayscale.
# Bentuk berubah dari (N, 28, 28) menjadi (N, 28, 28, 1).
x_train_original = x_train_original[..., np.newaxis]
x_test = x_test[..., np.newaxis]

In [ ]:
# 3. Acak indeks training saja.
# Seed tetap menghasilkan pembagian yang sama saat dijalankan ulang.
rng = np.random.default_rng(seed=42)
indices = rng.permutation(len(x_train_original))

n_train = int(0.9 * len(indices))
train_indices = indices[:n_train]
val_indices = indices[n_train:]

# Gambar dan label menggunakan indeks yang sama.
x_train = x_train_original[train_indices]
y_train = y_train_original[train_indices]

x_val = x_train_original[val_indices]
y_val = y_train_original[val_indices]

# Test tetap berasal dari pembagian resmi, tanpa diacak atau dicampur.

In [ ]:
# 4. Tampilkan jumlah, bentuk, tipe data, dan rentang piksel.
for nama, gambar in [
    ("Latihan", x_train),
    ("Validasi", x_val),
    ("Test", x_test),
]:
    print(
        f"{nama}: {len(gambar):,} gambar | "
        f"bentuk={gambar.shape} | dtype={gambar.dtype} | "
        f"piksel=[{gambar.min():.1f}, {gambar.max():.1f}]"
    )

In [ ]:
# 5. Tampilkan 10 contoh gambar latihan setelah preprocessing.
fig, axes = plt.subplots(2, 5, figsize=(10, 4))

for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i, :, :, 0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Label: {y_train[i]}")
    ax.axis("off")

fig.suptitle("Contoh MNIST setelah preprocessing")
plt.tight_layout()
plt.show()

In [ ]:
# 6. Buat PyTorch Dataset.
# Data sebelumnya masih berupa NumPy array dengan bentuk (N, 28, 28, 1).
# PyTorch menggunakan format channel-first, yaitu (N, C, H, W).
# Karena MNIST adalah gambar grayscale, jumlah kanalnya adalah 1.
class MNISTDataset(Dataset):
    def __init__(self, images):
        # Simpan kumpulan gambar yang sudah melalui preprocessing.
        self.images = images

    def __len__(self):
        # Kembalikan jumlah gambar dalam dataset.
        return len(self.images)

    def __getitem__(self, index):
        # Ambil satu gambar berdasarkan indeks.
        image = self.images[index]

        # Ubah NumPy array menjadi PyTorch tensor.
        image = torch.from_numpy(image)

        # Ubah bentuk dari HWC menjadi CHW.
        # (28, 28, 1) -> (1, 28, 28)
        image = image.permute(2, 0, 1)

        return image

In [ ]:
# 7. Buat dataset untuk training, validation, dan testing.

train_dataset = MNISTDataset(x_train)
val_dataset = MNISTDataset(x_val)
test_dataset = MNISTDataset(x_test)

print(f"Train      : {len(train_dataset):,} gambar")
print(f"Validasi   : {len(val_dataset):,} gambar")
print(f"Test       : {len(test_dataset):,} gambar")

In [ ]:
# 8. Periksa satu sample dari PyTorch Dataset.
# Pastikan bentuk, tipe data, dan rentang piksel sudah sesuai dengan kebutuhan model.

sample = train_dataset[0]

print(f"Shape : {sample.shape}")
print(f"Dtype : {sample.dtype}")
print(f"Min   : {sample.min().item():.1f}")
print(f"Max   : {sample.max().item():.1f}")

In [ ]:
# 9. Buat DataLoader untuk training, validation, dan testing.
# DataLoader digunakan untuk mengambil data dalam bentuk batch.
# Batch size = 128 berarti model akan menerima maksimal 128 gambar dalam satu batch.

BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=(device.type == "cuda"),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=(device.type == "cuda"),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=(device.type == "cuda"),
)

print(f"Batch size: {BATCH_SIZE}")

In [ ]:
# 10. Periksa satu batch dari training DataLoader.
# Bentuk yang diharapkan:
# (batch_size, channel, height, width)
# (128, 1, 28, 28)

images = next(iter(train_loader))

print(f"Shape : {images.shape}")
print(f"Dtype : {images.dtype}")
print(f"Min   : {images.min().item():.1f}")
print(f"Max   : {images.max().item():.1f}")

In [ ]:
# 11. Periksa jumlah batch pada setiap DataLoader.
# Jumlah batch digunakan untuk memastikan seluruh data dapat dibaca melalui DataLoader.

print(f"Jumlah batch train      : {len(train_loader)}")
print(f"Jumlah batch validation : {len(val_loader)}")
print(f"Jumlah batch test       : {len(test_loader)}")

In [ ]:

# 12. Membuat arsitektur Variational Autoencoder (VAE).
# Model VAE digunakan untuk mempelajari representasi laten dari gambar MNIST
# melalui encoder, reparameterization trick, dan decoder.

import torch
import torch.nn as nn
import torch.nn.functional as F


class VAE(nn.Module):
    """
    Variational Autoencoder (VAE) berbasis MLP untuk dataset MNIST.

    Parameters
    ----------
    hidden_dim : int
        Jumlah neuron pada hidden layer encoder dan decoder.
        Default: 128.

    latent_dim : int
        Dimensi latent space.
        Default: 2.
    """

    def __init__(self, hidden_dim=128, latent_dim=2):
        super().__init__()

        # Memastikan ukuran hidden layer dan latent space valid.
        if hidden_dim <= 0:
            raise ValueError("hidden_dim harus lebih besar dari 0.")

        if latent_dim <= 0:
            raise ValueError("latent_dim harus lebih besar dari 0.")

        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim

        # Encoder: mengubah gambar 784 piksel menjadi representasi tersembunyi.
        self.encoder_fc = nn.Linear(28 * 28, hidden_dim)

        # Menghasilkan parameter distribusi latent space, yaitu mu dan logvar.
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        # Decoder: mengubah latent vector menjadi representasi gambar.
        self.decoder_fc1 = nn.Linear(latent_dim, hidden_dim)
        self.decoder_fc2 = nn.Linear(hidden_dim, 28 * 28)

    def encode(self, x):
        """
        Mengubah gambar menjadi parameter distribusi latent space.

        Input:
            x: [B, 1, 28, 28]

        Output:
            mu:     [B, latent_dim]
            logvar: [B, latent_dim]
        """
        # Meratakan gambar dari [B, 1, 28, 28] menjadi [B, 784].
        x = torch.flatten(x, start_dim=1)

        # Mengekstraksi fitur menggunakan encoder.
        hidden = F.relu(self.encoder_fc(x))

        # Menghasilkan mean dan log-variance distribusi laten.
        mu = self.fc_mu(hidden)
        logvar = self.fc_logvar(hidden)

        return mu, logvar

    def reparameterize(self, mu, logvar):
        """
        Sampling latent vector menggunakan reparameterization trick.

        z = mu + exp(0.5 * logvar) * epsilon
        epsilon ~ N(0, I)
        """
        # Menghitung standar deviasi dari log-variance.
        std = torch.exp(0.5 * logvar)

        # Mengambil sampel acak dari distribusi normal standar.
        epsilon = torch.randn_like(std)

        # Membentuk latent vector menggunakan mean, standar deviasi,
        # dan sampel acak.
        z = mu + std * epsilon

        return z

    def decode(self, z):
        """
        Mengubah latent vector menjadi rekonstruksi gambar.

        Input:
            z: [B, latent_dim]

        Output:
            reconstruction_flat: [B, 784]
        """
        # Memproses latent vector melalui hidden layer decoder.
        hidden = F.relu(self.decoder_fc1(z))

        # Menghasilkan nilai piksel dalam rentang 0 sampai 1.
        reconstruction_flat = torch.sigmoid(
            self.decoder_fc2(hidden)
        )

        return reconstruction_flat

    def forward(self, x):
        """
        Menjalankan keseluruhan proses VAE.

        Input:
            x: [B, 1, 28, 28]

        Output:
            reconstruction: [B, 1, 28, 28]
            mu:             [B, latent_dim]
            logvar:         [B, latent_dim]
        """
        # Memastikan input memiliki bentuk gambar MNIST yang sesuai.
        if x.ndim != 4 or tuple(x.shape[1:]) != (1, 28, 28):
            raise ValueError(
                "Input harus berbentuk [B, 1, 28, 28], "
                f"tetapi menerima {tuple(x.shape)}."
            )

        batch_size = x.shape[0]

        # 1. Encoder: menghasilkan parameter distribusi laten.
        mu, logvar = self.encode(x)

        # 2. Reparameterization: mengambil sampel latent vector.
        z = self.reparameterize(mu, logvar)

        # 3. Decoder: merekonstruksi gambar dari latent vector.
        reconstruction_flat = self.decode(z)

        # 4. Mengembalikan bentuk output menjadi gambar [B, 1, 28, 28].
        reconstruction = reconstruction_flat.reshape(
            batch_size, 1, 28, 28
        )

        return reconstruction, mu, logvar


# Membuat model VAE baseline dengan hidden_dim 128 dan latent_dim 2.
vae_model = VAE(hidden_dim=128, latent_dim=2).to(device)

# Menampilkan struktur arsitektur model VAE.
print(vae_model)


In [ ]:

# 13. Menguji arsitektur VAE menggunakan data dari DataLoader.
# Pengujian dilakukan untuk memastikan bentuk output model sesuai,
# nilai yang dihasilkan valid, dan model dapat memproses batch
# dengan ukuran berbeda sebelum memasuki tahap training.

# Mengambil satu batch dari DataLoader yang sudah tersedia
batch = next(iter(train_loader))

# Mengambil gambar tanpa membuat Dataset atau DataLoader baru
if isinstance(batch, (tuple, list)):
    images = batch[0]

elif isinstance(batch, dict):
    image_key = next(
        (
            key
            for key in ("image", "images", "data", "input", "x")
            if key in batch
        ),
        None,
    )

    if image_key is None:
        raise KeyError(
            "Tidak ditemukan key gambar pada batch. "
            f"Key yang tersedia: {list(batch.keys())}"
        )

    images = batch[image_key]

else:
    images = batch


# Memastikan batch sesuai dengan format input model
if not torch.is_tensor(images):
    raise TypeError("Data gambar harus berupa PyTorch tensor.")

if images.ndim != 4 or tuple(images.shape[1:]) != (1, 28, 28):
    raise ValueError(
        "Bentuk batch harus [B, 1, 28, 28]. "
        f"Bentuk aktual: {tuple(images.shape)}"
    )


# Menggunakan perangkat yang dipilih pada cell awal.
vae_model = vae_model.to(device)
vae_model.eval()

images = images.to(
    device=device,
    dtype=torch.float32,
    non_blocking=(device.type == "cuda"),
)


# Pengujian pertama: satu batch dari train_loader
with torch.no_grad():
    reconstruction, mu, logvar = vae_model(images)

batch_size = images.shape[0]

assert reconstruction.shape == images.shape
assert mu.shape == (batch_size, 2)
assert logvar.shape == (batch_size, 2)

assert torch.isfinite(reconstruction).all()
assert torch.isfinite(mu).all()
assert torch.isfinite(logvar).all()

assert reconstruction.min().item() >= 0.0
assert reconstruction.max().item() <= 1.0

print("=== Pengujian Satu Batch ===")
print("Device:", device)
print("Input gambar       :", tuple(images.shape))
print("Reconstruction     :", tuple(reconstruction.shape))
print("Mu                 :", tuple(mu.shape))
print("Logvar             :", tuple(logvar.shape))
print("Rentang reconstruction:",
      f"{reconstruction.min().item():.4f} - "
      f"{reconstruction.max().item():.4f}")
print("Pengujian batch berhasil.")


# Pengujian kedua: batch dengan ukuran lebih kecil
# Tidak membuat atau mengubah DataLoader.
partial_size = min(7, batch_size)
partial_images = images[:partial_size]

with torch.no_grad():
    partial_reconstruction, partial_mu, partial_logvar = (
        vae_model(partial_images)
    )

assert partial_reconstruction.shape == (
    partial_size, 1, 28, 28
)
assert partial_mu.shape == (partial_size, 2)
assert partial_logvar.shape == (partial_size, 2)

print("\n=== Pengujian Batch Lebih Kecil ===")
print("Input gambar       :", tuple(partial_images.shape))
print("Reconstruction     :", tuple(partial_reconstruction.shape))
print("Mu                 :", tuple(partial_mu.shape))
print("Logvar             :", tuple(partial_logvar.shape))
print("Pengujian batch lebih kecil berhasil.")


In [ ]:
# 14. Implementasi fungsi loss Variational Autoencoder (VAE).
# Fungsi ini menggabungkan Binary Cross Entropy (BCE) sebagai reconstruction loss
# dan KL Divergence untuk mengatur distribusi laten agar mendekati distribusi
# normal standar. Loss dihitung per gambar, kemudian dirata-ratakan berdasarkan
# jumlah sampel dalam batch. Fungsi ini akan digunakan pada tahap training.

import torch
import torch.nn.functional as F


def vae_loss_function(reconstruction, x, mu, logvar, beta=1.0):
    """
    Menghitung loss Variational Autoencoder (VAE).

    Parameter:
        reconstruction: Gambar hasil rekonstruksi, [B, 1, 28, 28].
        x:              Gambar asli, [B, 1, 28, 28].
        mu:             Mean distribusi laten, [B, latent_dim].
        logvar:         Log variance distribusi laten, [B, latent_dim].
        beta:           Bobot KL Divergence, default 1.0.

    Return:
        total_loss: Rata-rata total loss per gambar dalam batch.
        bce_loss:   Rata-rata reconstruction loss per gambar.
        kl_loss:    Rata-rata KL Divergence per gambar.
    """

    # 1. Validasi bentuk tensor.
    if reconstruction.shape != x.shape:
        raise ValueError(
            "Bentuk reconstruction dan x harus sama. "
            f"Diperoleh {tuple(reconstruction.shape)} dan {tuple(x.shape)}."
        )

    if x.ndim != 4 or tuple(x.shape[1:]) != (1, 28, 28):
        raise ValueError(
            "x harus berbentuk [B, 1, 28, 28]."
        )

    if mu.ndim != 2 or logvar.shape != mu.shape:
        raise ValueError(
            "mu dan logvar harus memiliki bentuk [B, latent_dim] "
            "yang sama."
        )

    if mu.shape[0] != x.shape[0]:
        raise ValueError(
            "Ukuran batch mu dan logvar harus sama dengan batch gambar."
        )

    if beta < 0:
        raise ValueError("beta tidak boleh negatif.")

    # 2. Menghitung BCE untuk setiap piksel, lalu menjumlahkan 784 piksel
    # pada setiap gambar. Hasil akhirnya memiliki bentuk [B].
    bce_per_pixel = F.binary_cross_entropy(
        reconstruction,
        x,
        reduction="none"
    )

    bce_per_image = bce_per_pixel.flatten(start_dim=1).sum(dim=1)

    # 3. Menghitung KL Divergence untuk setiap dimensi laten,
    # lalu menjumlahkannya untuk setiap gambar. Hasilnya berbentuk [B].
    kl_per_dimension = -0.5 * (
        1 + logvar - mu.pow(2) - logvar.exp()
    )

    kl_per_image = kl_per_dimension.sum(dim=1)

    # 4. Menggabungkan BCE dan KL Divergence untuk setiap gambar.
    # Total loss kemudian dirata-ratakan berdasarkan jumlah gambar dalam batch.
    total_per_image = bce_per_image + beta * kl_per_image

    bce_loss = bce_per_image.mean()
    kl_loss = kl_per_image.mean()
    total_loss = total_per_image.mean()

    return total_loss, bce_loss, kl_loss


# 5. Memastikan fungsi loss berhasil didefinisikan.
assert callable(vae_loss_function)

print("Fungsi vae_loss_function berhasil dibuat.")
print("Komponen: BCE + beta * KL Divergence")
print("Beta default:", 1.0)


In [ ]:
# 15. Menguji fungsi loss VAE menggunakan satu batch dari train_loader.
# Pengujian memastikan total loss, BCE, dan KL Divergence menghasilkan
# nilai yang valid dan konsisten tanpa melakukan training model.

vae_model.eval()

batch = next(iter(train_loader))

if isinstance(batch, (tuple, list)):
    x_test = batch[0]
elif isinstance(batch, dict):
    image_key = next(
        key for key in ("image", "images", "data", "input", "x")
        if key in batch
    )
    x_test = batch[image_key]
else:
    x_test = batch

x_test = x_test.to(
    device=device, dtype=torch.float32, non_blocking=(device.type == "cuda")
)

with torch.no_grad():
    reconstruction_test, mu_test, logvar_test = vae_model(x_test)

    total_loss_test, bce_test, kl_test = vae_loss_function(
        reconstruction_test,
        x_test,
        mu_test,
        logvar_test
    )

assert total_loss_test.ndim == 0
assert bce_test.ndim == 0
assert kl_test.ndim == 0

assert torch.isfinite(total_loss_test).item()
assert torch.isfinite(bce_test).item()
assert torch.isfinite(kl_test).item()

assert torch.allclose(
    total_loss_test,
    bce_test + kl_test,
    rtol=1e-5,
    atol=1e-6
)

print("\n=== Pemeriksaan VAE Loss ===")
print(f"Batch size       : {x_test.shape[0]}")
print(f"Reconstruction BCE: {bce_test.item():.4f}")
print(f"KL Divergence     : {kl_test.item():.4f}")
print(f"Total VAE Loss    : {total_loss_test.item():.4f}")
print("Pemeriksaan loss berhasil.")


In [ ]:
# 16. Menyiapkan konfigurasi training baseline dan optimizer.
# Cell ini menentukan hyperparameter awal, membuat instance VAE baru,
# dan menyiapkan optimizer Adam sebelum proses training dimulai.
# Pemeriksaan dilakukan untuk memastikan konfigurasi model, optimizer,
# fungsi loss, dan DataLoader sudah sesuai.

import torch
import torch.optim as optim


# ============================================================
# 1. Konfigurasi baseline
# ============================================================

HIDDEN_DIM = 128
LATENT_DIM = 2
BATCH_SIZE = 128
LEARNING_RATE = 0.001
BETA = 1.0

# Gunakan perangkat GPU/CPU yang dipilih pada cell awal.


# ============================================================
# 2. Membuat instance model untuk training baseline
# ============================================================

# Menggunakan kelas VAE dari Cell 12.
# Instance baru memastikan training dimulai dari bobot awal
# yang terpisah dari pengujian model sebelumnya.

vae_model = VAE(
    hidden_dim=HIDDEN_DIM,
    latent_dim=LATENT_DIM
).to(device)


# ============================================================
# 3. Menyiapkan optimizer Adam
# ============================================================

# Adam memperbarui parameter model berdasarkan gradien loss.
# Parameter model belum diperbarui sampai training loop dijalankan.

optimizer = optim.Adam(
    vae_model.parameters(),
    lr=LEARNING_RATE
)


# ============================================================
# 4. Pemeriksaan konfigurasi
# ============================================================

# Memastikan instance model menggunakan konfigurasi baseline.
assert vae_model.hidden_dim == HIDDEN_DIM
assert vae_model.latent_dim == LATENT_DIM

# Memastikan seluruh parameter model berada pada perangkat terpilih.
assert all(
    parameter.device.type == device.type
    for parameter in vae_model.parameters()
)

# Memastikan optimizer menggunakan learning rate yang ditentukan.
for param_group in optimizer.param_groups:
    assert param_group["lr"] == LEARNING_RATE

# Memastikan fungsi loss dari Cell 14 tersedia.
assert callable(vae_loss_function)

# Memastikan DataLoader yang sudah ada tetap tersedia.
assert train_loader is not None
assert val_loader is not None
assert test_loader is not None

# Memastikan batch size train_loader sesuai konfigurasi baseline.
assert train_loader.batch_size == BATCH_SIZE, (
    f"Batch size train_loader adalah {train_loader.batch_size}, "
    f"sedangkan konfigurasi baseline adalah {BATCH_SIZE}."
)


# ============================================================
# 5. Menampilkan ringkasan konfigurasi
# ============================================================

print("=== Konfigurasi Training Baseline ===")
print(f"Hidden dimension : {HIDDEN_DIM}")
print(f"Latent dimension : {LATENT_DIM}")
print(f"Batch size       : {BATCH_SIZE}")
print(f"Learning rate    : {LEARNING_RATE}")
print(f"Beta             : {BETA}")
print(f"Optimizer        : {type(optimizer).__name__}")
print(f"Device           : {device}")
print(f"Loss function    : {vae_loss_function.__name__}")

print("\nJumlah parameter model:",
      sum(p.numel() for p in vae_model.parameters()))

print("Jumlah parameter yang dapat dilatih:",
      sum(p.numel() for p in vae_model.parameters()
          if p.requires_grad))

print("\nPemeriksaan konfigurasi berhasil.")


In [ ]:
# 17. Fungsi training dan evaluasi VAE
# Menggunakan VAE, vae_loss_function, train_loader, val_loader,
# optimizer, device, dan konfigurasi yang sudah tersedia dari Cell 12–16.
# Fungsi ini dapat digunakan kembali untuk konfigurasi tuning berikutnya.

import torch


def train_one_epoch(
    model,
    data_loader,
    optimizer,
    device,
    beta=1.0,
):
    """Melatih model selama satu epoch dan mengembalikan rata-rata loss."""

    model.train()

    total_loss_sum = 0.0
    total_bce_sum = 0.0
    total_kl_sum = 0.0
    total_samples = 0

    for batch in data_loader:
        # Mendukung batch berbentuk tensor, tuple/list, atau dictionary.
        if isinstance(batch, (tuple, list)):
            images = batch[0]
        elif isinstance(batch, dict):
            if "image" in batch:
                images = batch["image"]
            elif "images" in batch:
                images = batch["images"]
            else:
                raise KeyError(
                    "Batch dictionary harus memiliki key 'image' atau 'images'."
                )
        else:
            images = batch

        images = images.to(device, non_blocking=(device.type == "cuda"))

        # Bersihkan gradien dari langkah sebelumnya.
        optimizer.zero_grad()

        # Forward pass: menghasilkan rekonstruksi, mu, dan logvar.
        reconstruction, mu, logvar = model(images)

        # Hitung total loss, reconstruction loss (BCE), dan KL divergence.
        loss, bce_loss, kl_loss = vae_loss_function(
            reconstruction,
            images,
            mu,
            logvar,
            beta=beta,
        )

        # Backpropagation dan pembaruan parameter model.
        loss.backward()
        optimizer.step()

        # Akumulasi loss berbobot jumlah sampel agar rata-rata epoch akurat.
        batch_size_actual = images.size(0)
        total_loss_sum += loss.item() * batch_size_actual
        total_bce_sum += bce_loss.item() * batch_size_actual
        total_kl_sum += kl_loss.item() * batch_size_actual
        total_samples += batch_size_actual

    if total_samples == 0:
        raise ValueError("DataLoader training tidak memiliki sampel.")

    return {
        "loss": total_loss_sum / total_samples,
        "bce": total_bce_sum / total_samples,
        "kl": total_kl_sum / total_samples,
    }


def evaluate_vae(
    model,
    data_loader,
    device,
    beta=1.0,
):
    """Mengevaluasi VAE tanpa memperbarui parameter model."""

    model.eval()

    total_loss_sum = 0.0
    total_bce_sum = 0.0
    total_kl_sum = 0.0
    total_samples = 0

    with torch.no_grad():
        for batch in data_loader:
            # Ambil citra dari format batch yang didukung.
            if isinstance(batch, (tuple, list)):
                images = batch[0]
            elif isinstance(batch, dict):
                if "image" in batch:
                    images = batch["image"]
                elif "images" in batch:
                    images = batch["images"]
                else:
                    raise KeyError(
                        "Batch dictionary harus memiliki key 'image' atau 'images'."
                    )
            else:
                images = batch

            images = images.to(device, non_blocking=(device.type == "cuda"))

            # Forward pass tanpa menghitung gradien.
            reconstruction, mu, logvar = model(images)

            loss, bce_loss, kl_loss = vae_loss_function(
                reconstruction,
                images,
                mu,
                logvar,
                beta=beta,
            )

            batch_size_actual = images.size(0)
            total_loss_sum += loss.item() * batch_size_actual
            total_bce_sum += bce_loss.item() * batch_size_actual
            total_kl_sum += kl_loss.item() * batch_size_actual
            total_samples += batch_size_actual

    if total_samples == 0:
        raise ValueError("DataLoader validation tidak memiliki sampel.")

    return {
        "loss": total_loss_sum / total_samples,
        "bce": total_bce_sum / total_samples,
        "kl": total_kl_sum / total_samples,
    }


# Jalankan satu epoch baseline sebagai pengujian awal.
train_metrics = train_one_epoch(
    model=vae_model,
    data_loader=train_loader,
    optimizer=optimizer,
    device=device,
    beta=BETA,
)

# Evaluasi validation loss setelah satu epoch training.
val_metrics = evaluate_vae(
    model=vae_model,
    data_loader=val_loader,
    device=device,
    beta=BETA,
)

# Pastikan hasil evaluasi berupa angka yang valid.
for metric_name, metric_value in train_metrics.items():
    assert torch.isfinite(torch.tensor(metric_value)).item(), (
        f"Training metric {metric_name} tidak valid: {metric_value}"
    )

for metric_name, metric_value in val_metrics.items():
    assert torch.isfinite(torch.tensor(metric_value)).item(), (
        f"Validation metric {metric_name} tidak valid: {metric_value}"
    )

# Tampilkan ringkasan hasil epoch pertama.
print("=== Baseline VAE: Epoch 1 ===")
print(f"Training loss      : {train_metrics['loss']:.4f}")
print(f"Training BCE       : {train_metrics['bce']:.4f}")
print(f"Training KL        : {train_metrics['kl']:.4f}")
print(f"Validation loss    : {val_metrics['loss']:.4f}")
print(f"Validation BCE     : {val_metrics['bce']:.4f}")
print(f"Validation KL      : {val_metrics['kl']:.4f}")
print(f"Training samples   : {len(train_loader.dataset)}")
print(f"Validation samples : {len(val_loader.dataset)}")
print("\nTraining satu epoch dan evaluasi validation berhasil.")


In [ ]:
# 18. Training baseline VAE selama tepat 10 epoch
# Model dimulai dari bobot baru, bukan bobot hasil pengujian Cell 17.
# Dataset, DataLoader, arsitektur VAE, dan fungsi loss tetap menggunakan
# komponen yang sudah dibuat pada Cell 1–17.

import torch
import torch.optim as optim
import pandas as pd
import random
import numpy as np

# 1. Tetapkan seed agar inisialisasi dan eksperimen lebih mudah direproduksi.
# Seed tidak menjamin hasil identik pada semua lingkungan/perangkat.
BASELINE_SEED = 42

random.seed(BASELINE_SEED)
np.random.seed(BASELINE_SEED)
torch.manual_seed(BASELINE_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BASELINE_SEED)

# 2. Inisialisasi model baru dengan konfigurasi baseline yang sama.
# Model ini tidak menggunakan bobot hasil pengujian satu epoch pada Cell 17.
baseline_model = VAE(
    hidden_dim=HIDDEN_DIM,
    latent_dim=LATENT_DIM,
).to(device)

# Buat optimizer baru yang terhubung ke parameter model baseline baru.
baseline_optimizer = optim.Adam(
    baseline_model.parameters(),
    lr=LEARNING_RATE,
)

# 3. Siapkan tempat penyimpanan metrik setiap epoch.
# Dictionary ini dapat diubah menjadi DataFrame untuk tabel dan grafik.
baseline_history = {
    "epoch": [],
    "train_loss": [],
    "train_bce": [],
    "train_kl": [],
    "val_loss": [],
    "val_bce": [],
    "val_kl": [],
}

# 4. Jalankan tepat 10 epoch training dan evaluasi validation.
NUM_EPOCHS = 10

for epoch in range(1, NUM_EPOCHS + 1):
    # Training hanya menggunakan train_loader.
    train_metrics = train_one_epoch(
        model=baseline_model,
        data_loader=train_loader,
        optimizer=baseline_optimizer,
        device=device,
        beta=BETA,
    )

    # Evaluasi menggunakan val_loader, bukan test_loader.
    val_metrics = evaluate_vae(
        model=baseline_model,
        data_loader=val_loader,
        device=device,
        beta=BETA,
    )

    # Simpan semua metrik untuk analisis, tabel, dan grafik selanjutnya.
    baseline_history["epoch"].append(epoch)
    baseline_history["train_loss"].append(train_metrics["loss"])
    baseline_history["train_bce"].append(train_metrics["bce"])
    baseline_history["train_kl"].append(train_metrics["kl"])
    baseline_history["val_loss"].append(val_metrics["loss"])
    baseline_history["val_bce"].append(val_metrics["bce"])
    baseline_history["val_kl"].append(val_metrics["kl"])

    # Tampilkan hasil epoch saat ini.
    print(
        f"Epoch [{epoch:02d}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_metrics['loss']:.4f} | "
        f"Train BCE: {train_metrics['bce']:.4f} | "
        f"Train KL: {train_metrics['kl']:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val BCE: {val_metrics['bce']:.4f} | "
        f"Val KL: {val_metrics['kl']:.4f}"
    )

# 5. Ubah riwayat training menjadi tabel.
baseline_results = pd.DataFrame(baseline_history)

# 6. Periksa kelengkapan dan validitas hasil.
assert len(baseline_results) == NUM_EPOCHS, (
    "Jumlah epoch pada hasil tidak sesuai."
)

assert baseline_results["epoch"].tolist() == list(
    range(1, NUM_EPOCHS + 1)
), "Urutan epoch tidak sesuai."

metric_columns = [
    "train_loss", "train_bce", "train_kl",
    "val_loss", "val_bce", "val_kl",
]

assert np.isfinite(
    baseline_results[metric_columns].to_numpy()
).all(), "Ada metrik yang bukan angka finite."

# Periksa konsistensi total loss dengan BCE + beta * KL.
assert np.allclose(
    baseline_results["train_loss"],
    baseline_results["train_bce"] + BETA * baseline_results["train_kl"],
    rtol=1e-4,
    atol=1e-3,
), "Training loss tidak konsisten dengan BCE + beta * KL."

assert np.allclose(
    baseline_results["val_loss"],
    baseline_results["val_bce"] + BETA * baseline_results["val_kl"],
    rtol=1e-4,
    atol=1e-3,
), "Validation loss tidak konsisten dengan BCE + beta * KL."

# 7. Tampilkan tabel hasil lengkap dan konfigurasi baseline.
print("\n=== Hasil Training Baseline VAE ===")
print(baseline_results.round(4).to_string(index=False))

print("\n=== Konfigurasi Baseline ===")
print(f"Seed          : {BASELINE_SEED}")
print(f"Hidden dim    : {HIDDEN_DIM}")
print(f"Latent dim    : {LATENT_DIM}")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Learning rate : {LEARNING_RATE}")
print(f"Beta          : {BETA}")
print(f"Epoch         : {NUM_EPOCHS}")
print(f"Device        : {device}")

print("\nTraining baseline 10 epoch selesai.")
print("Seluruh metrik berhasil diperiksa.")


In [ ]:
# 19. Menyimpan model baseline dan seluruh hasil eksperimen
# Menggunakan baseline_model, baseline_optimizer, baseline_history,
# baseline_results, dan konfigurasi yang sudah tersedia dari Cell 16–18.
# Tidak menjalankan training ulang.

from pathlib import Path
import json
import torch
import pandas as pd

# 1. Buat folder output jika belum tersedia.
# Path relatif terhadap working directory notebook saat ini.
output_dir = Path("results") / "baseline"
output_dir.mkdir(parents=True, exist_ok=True)

# 2. Simpan konfigurasi eksperimen baseline.
baseline_config = {
    "experiment_name": "vae_baseline",
    "model_name": "VAE",
    "hidden_dim": HIDDEN_DIM,
    "latent_dim": LATENT_DIM,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "beta": BETA,
    "num_epochs": NUM_EPOCHS,
    "seed": BASELINE_SEED,
    "optimizer": type(baseline_optimizer).__name__,
    "loss_function": "vae_loss_function",
    "device_used": str(device),
    "training_samples": len(train_loader.dataset),
    "validation_samples": len(val_loader.dataset),
    "validation_source": "val_loader",
}

# 3. Ambil metrik epoch terakhir dengan benar.
# baseline_history berisi list metrik dari epoch 1 sampai epoch 10.
final_train_metrics = {
    key: float(value[-1])
    for key, value in baseline_history.items()
    if key.startswith("train_")
}

final_validation_metrics = {
    key: float(value[-1])
    for key, value in baseline_history.items()
    if key.startswith("val_")
}

# Pastikan metrik epoch terakhir tersedia dan valid.
assert final_train_metrics, "Metrik training tidak ditemukan."
assert final_validation_metrics, "Metrik validation tidak ditemukan."

assert all(
    torch.isfinite(torch.tensor(value)).item()
    for value in final_train_metrics.values()
), "Terdapat metrik training yang tidak valid."

assert all(
    torch.isfinite(torch.tensor(value)).item()
    for value in final_validation_metrics.values()
), "Terdapat metrik validation yang tidak valid."

# 4. Simpan checkpoint: bobot model, konfigurasi, dan metrik akhir.
checkpoint_path = output_dir / "vae_baseline.pth"

torch.save(
    {
        "model_state_dict": baseline_model.state_dict(),
        "config": baseline_config,
        "final_train_metrics": final_train_metrics,
        "final_validation_metrics": final_validation_metrics,
    },
    checkpoint_path,
)

# 5. Simpan konfigurasi baseline dalam format JSON.
config_path = output_dir / "vae_baseline_config.json"

with open(config_path, "w", encoding="utf-8") as file:
    json.dump(baseline_config, file, indent=4, ensure_ascii=False)

# 6. Simpan riwayat metrik lengkap dari seluruh epoch ke JSON.
# Konversi nilai menjadi tipe Python standar agar kompatibel dengan JSON.
history_json = {
    key: [
        int(value) if key == "epoch" else float(value)
        for value in values
    ]
    for key, values in baseline_history.items()
}

history_path = output_dir / "vae_baseline_history.json"

with open(history_path, "w", encoding="utf-8") as file:
    json.dump(history_json, file, indent=4, ensure_ascii=False)

# 7. Simpan tabel hasil baseline ke CSV.
csv_path = output_dir / "vae_baseline_results.csv"
baseline_results.to_csv(csv_path, index=False)

# 8. Periksa keberadaan dan ukuran semua file yang dibuat.
saved_files = [
    checkpoint_path,
    config_path,
    history_path,
    csv_path,
]

for file_path in saved_files:
    assert file_path.is_file(), f"File tidak ditemukan: {file_path}"
    assert file_path.stat().st_size > 0, f"File kosong: {file_path}"

# 9. Baca kembali checkpoint untuk memverifikasi isinya.
# weights_only=True didukung oleh versi PyTorch modern,
# termasuk versi yang mendukung format checkpoint berisi tensor dan tipe dasar.
try:
    saved_checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True,
    )
except TypeError:
    # Fallback untuk versi PyTorch lama yang belum memiliki argumen weights_only.
    saved_checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
    )

assert "model_state_dict" in saved_checkpoint
assert "config" in saved_checkpoint
assert "final_train_metrics" in saved_checkpoint
assert "final_validation_metrics" in saved_checkpoint

assert saved_checkpoint["config"]["num_epochs"] == 10
assert saved_checkpoint["config"]["hidden_dim"] == HIDDEN_DIM
assert saved_checkpoint["config"]["latent_dim"] == LATENT_DIM

# 10. Validasi CSV: harus berisi tepat 10 epoch.
saved_results = pd.read_csv(csv_path)

assert len(saved_results) == 10, (
    f"CSV seharusnya berisi 10 baris, tetapi ditemukan {len(saved_results)}."
)

assert "epoch" in saved_results.columns
assert saved_results["epoch"].tolist() == list(range(1, 11))

# 11. Pastikan riwayat JSON juga memuat 10 epoch.
with open(history_path, "r", encoding="utf-8") as file:
    saved_history = json.load(file)

assert len(saved_history["epoch"]) == 10
assert saved_history["epoch"] == list(range(1, 11))

# 12. Tampilkan lokasi dan ringkasan hasil penyimpanan.
print("=== Penyimpanan Baseline Berhasil ===")
print(f"Folder output: {output_dir.resolve()}")

print("\nFile yang berhasil disimpan:")
for file_path in saved_files:
    print(f"- {file_path.resolve()}")
    print(f"  Ukuran: {file_path.stat().st_size:,} bytes")

print("\nJumlah epoch dalam CSV:", len(saved_results))
print("Jumlah epoch dalam JSON:", len(saved_history["epoch"]))

print("\nMetrik training epoch terakhir:")
for name, value in final_train_metrics.items():
    print(f"- {name}: {value:.4f}")

print("\nMetrik validation epoch terakhir:")
for name, value in final_validation_metrics.items():
    print(f"- {name}: {value:.4f}")

print("\nSemua pemeriksaan penyimpanan berhasil.")


In [ ]:

# 20. Persiapan Grid Search

from itertools import product
from pathlib import Path
import json

# Daftar hyperparameter
GRID_PARAMS = {
    "hidden_dim": [256, 512, 64],
    "latent_dim": [4, 8, 16],
    "batch_size": [32, 64],
    "learning_rate": [0.1, 0.01, 0.005],
}

GRID_EPOCHS = 10
GRID_SEED = 42
GRID_BETA = 1.0

# Folder khusus Grid Search agar hasil T1-T5 tetap aman
GRID_SEARCH_DIR = Path("results/grid_search")
GRID_SEARCH_DIR.mkdir(parents=True, exist_ok=True)

# Membentuk seluruh kombinasi secara otomatis
param_names = list(GRID_PARAMS.keys())

GRID_COMBINATIONS = [
    dict(zip(param_names, values))
    for values in product(
        *(GRID_PARAMS[name] for name in param_names)
    )
]

# Menyimpan konfigurasi grid
grid_config = {
    "parameters": GRID_PARAMS,
    "epochs": GRID_EPOCHS,
    "seed": GRID_SEED,
    "beta": GRID_BETA,
    "total_combinations": len(GRID_COMBINATIONS),
}

with open(
    GRID_SEARCH_DIR / "grid_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(grid_config, f, indent=4)

print("=" * 45)
print("PERSIAPAN GRID SEARCH")
print("=" * 45)
print(f"Hidden dimensions : {GRID_PARAMS['hidden_dim']}")
print(f"Latent dimensions : {GRID_PARAMS['latent_dim']}")
print(f"Batch sizes       : {GRID_PARAMS['batch_size']}")
print(f"Learning rates    : {GRID_PARAMS['learning_rate']}")
print(f"Epoch per model   : {GRID_EPOCHS}")
print(f"Total kombinasi   : {len(GRID_COMBINATIONS)}")
print(f"Folder hasil      : {GRID_SEARCH_DIR}")

print("\nContoh 5 kombinasi pertama:")
for i, config in enumerate(GRID_COMBINATIONS[:5], start=1):
    print(f"{i}. {config}")

assert len(GRID_COMBINATIONS) == 54
print("\nPemeriksaan berhasil: terdapat 54 kombinasi.")


In [ ]:
# 21. Menjalankan Grid Search untuk VAE

import json
import time
import random
import copy
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import DataLoader

# --------------------------------------------------
# 1. Persiapan folder dan file hasil
# --------------------------------------------------

GRID_SEARCH_DIR = Path("results/grid_search")
GRID_SEARCH_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_PATH = GRID_SEARCH_DIR / "grid_search_summary.csv"

RESULT_COLUMNS = [
    "config_id",
    "hidden_dim",
    "latent_dim",
    "batch_size",
    "learning_rate",
    "epochs",
    "best_val_loss",
    "best_val_bce",
    "best_val_kl",
    "best_val_epoch",
    "final_train_loss",
    "duration_seconds",
    "status",
    "error",
]

if SUMMARY_PATH.exists():
    summary_df = pd.read_csv(SUMMARY_PATH)

    # Pastikan kolom sesuai dengan format yang digunakan
    for col in RESULT_COLUMNS:
        if col not in summary_df.columns:
            summary_df[col] = np.nan
else:
    summary_df = pd.DataFrame(columns=RESULT_COLUMNS)


def save_summary_row(result):
    """Simpan atau perbarui hasil satu kombinasi."""
    global summary_df

    config_id = result["config_id"]

    summary_df = summary_df[
        summary_df["config_id"] != config_id
    ]

    summary_df = pd.concat(
        [summary_df, pd.DataFrame([result])],
        ignore_index=True,
    )

    summary_df.to_csv(SUMMARY_PATH, index=False)


# --------------------------------------------------
# 2. Siapkan DataLoader berdasarkan batch size
# --------------------------------------------------

def make_grid_loader(dataset, batch_size, shuffle, seed):
    generator = torch.Generator()
    generator.manual_seed(seed)

    # Pertahankan pengaturan worker dari DataLoader awal
    original_loader = (
        train_loader if shuffle else val_loader
    )

    kwargs = {
        "dataset": dataset,
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": original_loader.num_workers,
        "pin_memory": original_loader.pin_memory,
        "drop_last": False,
        "generator": generator if shuffle else None,
    }

    if original_loader.num_workers > 0:
        kwargs["persistent_workers"] = (
            original_loader.persistent_workers
        )

        if original_loader.prefetch_factor is not None:
            kwargs["prefetch_factor"] = (
                original_loader.prefetch_factor
            )

    return DataLoader(**kwargs)


# --------------------------------------------------
# 3. Jalankan satu kombinasi hyperparameter
# --------------------------------------------------

def run_grid_combination(config, combination_number):
    config_id = f"GS_{combination_number:03d}"
    output_dir = GRID_SEARCH_DIR / config_id
    output_dir.mkdir(parents=True, exist_ok=True)

    config = dict(config)

    config["epochs"] = GRID_EPOCHS
    config["beta"] = GRID_BETA
    config["seed"] = GRID_SEED

    config_path = output_dir / "config.json"

    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=4)

    # Set seed agar inisialisasi model dapat direproduksi
    seed = GRID_SEED
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    train_loader_grid = make_grid_loader(
        train_loader.dataset,
        config["batch_size"],
        shuffle=True,
        seed=seed,
    )

    val_loader_grid = make_grid_loader(
        val_loader.dataset,
        config["batch_size"],
        shuffle=False,
        seed=seed,
    )

    model = VAE(
        config["hidden_dim"],
        config["latent_dim"],
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
    )

    history = []
    best_val_loss = float("inf")
    best_val_bce = float("nan")
    best_val_kl = float("nan")
    best_val_epoch = 0
    final_train_loss = float("nan")

    start_time = time.time()
    status = "completed"
    error_message = ""

    print("\n" + "=" * 65)
    print(
        f"{config_id} | "
        f"hidden={config['hidden_dim']}, "
        f"latent={config['latent_dim']}, "
        f"batch={config['batch_size']}, "
        f"lr={config['learning_rate']}"
    )
    print("=" * 65)

    try:
        for epoch in range(1, GRID_EPOCHS + 1):

            train_metrics = train_one_epoch(
                model,
                train_loader_grid,
                optimizer,
                device,
                beta=GRID_BETA,
            )

            val_metrics = evaluate_vae(
                model,
                val_loader_grid,
                device,
                beta=GRID_BETA,
            )

            metrics = {
                "epoch": epoch,
                "train_loss": float(train_metrics["loss"]),
                "train_bce": float(train_metrics["bce"]),
                "train_kl": float(train_metrics["kl"]),
                "val_loss": float(val_metrics["loss"]),
                "val_bce": float(val_metrics["bce"]),
                "val_kl": float(val_metrics["kl"]),
            }

            # Hentikan kombinasi jika hasil tidak valid
            if not all(
                np.isfinite(value)
                for key, value in metrics.items()
                if key != "epoch"
            ):
                raise ValueError(
                    f"Loss NaN/Inf pada epoch {epoch}"
                )

            history.append(metrics)
            final_train_loss = metrics["train_loss"]

            if metrics["val_loss"] < best_val_loss:
                best_val_loss = metrics["val_loss"]
                best_val_bce = metrics["val_bce"]
                best_val_kl = metrics["val_kl"]
                best_val_epoch = epoch

                torch.save(
                    {
                        "model_state_dict": copy.deepcopy(
                            model.state_dict()
                        ),
                        "config": config,
                        "best_val_loss": best_val_loss,
                        "best_val_epoch": best_val_epoch,
                    },
                    output_dir / "best_model.pth",
                )

            # Simpan progres setelah setiap epoch
            with open(
                output_dir / "history.json",
                "w",
                encoding="utf-8",
            ) as f:
                json.dump(history, f, indent=4)

            pd.DataFrame(history).to_csv(
                output_dir / "epoch_history.csv",
                index=False,
            )

            print(
                f"Epoch {epoch:02d}/{GRID_EPOCHS} | "
                f"Train Loss: {metrics['train_loss']:.4f} | "
                f"Val Loss: {metrics['val_loss']:.4f} | "
                f"BCE: {metrics['val_bce']:.4f} | "
                f"KL: {metrics['val_kl']:.4f}"
            )

    except Exception as exc:
        status = "failed"
        error_message = f"{type(exc).__name__}: {exc}"
        print(f"❌ {config_id} gagal: {error_message}")

    duration = time.time() - start_time

    result = {
        "config_id": config_id,
        "hidden_dim": config["hidden_dim"],
        "latent_dim": config["latent_dim"],
        "batch_size": config["batch_size"],
        "learning_rate": config["learning_rate"],
        "epochs": GRID_EPOCHS,
        "best_val_loss": (
            best_val_loss if np.isfinite(best_val_loss)
            else np.nan
        ),
        "best_val_bce": (
            best_val_bce if np.isfinite(best_val_bce)
            else np.nan
        ),
        "best_val_kl": (
            best_val_kl if np.isfinite(best_val_kl)
            else np.nan
        ),
        "best_val_epoch": best_val_epoch,
        "final_train_loss": (
            final_train_loss
            if np.isfinite(final_train_loss)
            else np.nan
        ),
        "duration_seconds": round(duration, 2),
        "status": status,
        "error": error_message,
    }

    save_summary_row(result)

    print(
        f"Status: {status} | "
        f"Best validation loss: {result['best_val_loss']}"
    )

    return result


# --------------------------------------------------
# 4. Jalankan seluruh kombinasi
# --------------------------------------------------

completed_ids = set(
    summary_df.loc[
        summary_df["status"] == "completed",
        "config_id",
    ].dropna().astype(str)
)

remaining = [
    (i, config)
    for i, config in enumerate(
        GRID_COMBINATIONS, start=1
    )
    if f"GS_{i:03d}" not in completed_ids
]

print("=" * 65)
print("MEMULAI GRID SEARCH")
print(f"Total kombinasi       : {len(GRID_COMBINATIONS)}")
print(f"Kombinasi sudah selesai: {len(completed_ids)}")
print(f"Kombinasi tersisa     : {len(remaining)}")
print(f"Perangkat             : {device}")
print(f"Folder hasil          : {GRID_SEARCH_DIR}")
print("=" * 65)

for position, (number, config) in enumerate(remaining, start=1):
    print(
        f"\nProgres kombinasi: {position}/{len(remaining)}"
    )

    run_grid_combination(config, number)

# --------------------------------------------------
# 5. Ringkasan hasil
# --------------------------------------------------

summary_df = pd.read_csv(SUMMARY_PATH)

successful = summary_df[
    (summary_df["status"] == "completed")
    & summary_df["best_val_loss"].notna()
].copy()

successful = successful.sort_values(
    "best_val_loss",
    ascending=True,
)

print("\n" + "=" * 65)
print("GRID SEARCH SELESAI")
print("=" * 65)
print(f"Total tercatat : {len(summary_df)}")
print(f"Berhasil       : {(summary_df['status'] == 'completed').sum()}")
print(f"Gagal          : {(summary_df['status'] == 'failed').sum()}")

if not successful.empty:
    print("\n5 kombinasi terbaik berdasarkan validation loss:")
    print(
        successful[
            [
                "config_id",
                "hidden_dim",
                "latent_dim",
                "batch_size",
                "learning_rate",
                "best_val_loss",
                "best_val_epoch",
            ]
        ].head(5).to_string(index=False)
    )

    print("\nKombinasi terbaik:")
    print(successful.iloc[0].to_string())

print(f"\nFile ringkasan: {SUMMARY_PATH}")
